In [25]:
!pip install --upgrade certifi

In [26]:
import requests
import zipfile
import io
import os

url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"

response = requests.get(url, timeout=60)

print("Status code:", response.status_code)
print("File size:", len(response.content), "bytes")
print("Content type:", response.headers.get("content-type"))

if response.status_code == 200 and zipfile.is_zipfile(io.BytesIO(response.content)):
    with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
        zip_ref.extractall("/content/")

    print("Dataset extracted successfully!")
    print("Ratings file exists:", os.path.exists("/content/ml-100k/u.data"))
else:
    print("The download did not return a valid ZIP file.")
    print("Response preview:", response.text[:300])

Status code: 200
File size: 4924029 bytes
Content type: application/zip
Dataset extracted successfully!
Ratings file exists: True


In [27]:
import pandas as pd

ratings = pd.read_csv(
    "/content/ml-100k/u.data",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

print("Number of ratings:", len(ratings))
print("Number of users:", ratings["user_id"].nunique())
print("Number of movies rated:", ratings["movie_id"].nunique())

display(ratings.head())

Number of ratings: 100000
Number of users: 943
Number of movies rated: 1682


,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [28]:
rating_matrix = ratings.pivot_table(
    index="user_id",
    columns="movie_id",
    values="rating"
)

display(rating_matrix.head())

movie_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:

user_similarity = rating_matrix.T.corr(
    method="pearson",
    min_periods=5
)

display(user_similarity.iloc[:5, :5].round(2))

user_id,1,2,3,4,5
user_id,,,,,
1,1.00,0.16,0.11,0.50,0.42
2,0.16,1.00,0.07,0.15,0.33
3,0.11,0.07,1.00,-0.26,NaN
4,0.50,0.15,-0.26,1.00,NaN
5,0.42,0.33,NaN,NaN,1.00


In [30]:
similar_users = user_similarity.loc[1].drop(labels=[1])

similar_users = similar_users.dropna().sort_values(ascending=False)

print("Users most similar to User 1:")
display(similar_users.head(10))

Users most similar to User 1:


,1
user_id,
520,0.918559
107,0.916534
105,0.882299
510,0.862582
803,0.849655
791,0.845489
572,0.843661
691,0.838742
754,0.810643


In [31]:

user_520_ratings = rating_matrix.loc[520].dropna()

user_520_ratings = user_520_ratings.sort_values(ascending=False)

print("Movies rated by User 520:", len(user_520_ratings))
print("\nUser 520's highest-rated movies:")

display(user_520_ratings.head(10))

Movies rated by User 520: 23

User 520's highest-rated movies:


,520
movie_id,
242,5.0
269,5.0
286,5.0
690,5.0
898,5.0
283,4.0
25,4.0
315,4.0
990,4.0


In [32]:

liked_movies = user_520_ratings[user_520_ratings >= 4]

user_1_movies = rating_matrix.loc[1].dropna().index

recommendable_movies = liked_movies.drop(
    labels=user_1_movies,
    errors="ignore"
)

print("Movies User 520 liked:", len(liked_movies))
print("Movies User 1 has already rated:", len(user_1_movies))
print("Potential recommendations:", len(recommendable_movies))

display(recommendable_movies)

Movies User 520 liked: 13
Movies User 1 has already rated: 272
Potential recommendations: 9


,520
movie_id,
286,5.0
690,5.0
898,5.0
283,4.0
315,4.0
990,4.0
289,4.0
300,4.0
310,4.0


In [33]:

movie_titles = pd.read_csv(
    "/content/ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    usecols=[0, 1],
    names=["movie_id", "title"]
)

potential_recommendations = recommendable_movies.rename(
    "user_520_rating"
).reset_index()

potential_recommendations = potential_recommendations.merge(
    movie_titles,
    on="movie_id",
    how="left"
)

display(
    potential_recommendations[
        ["movie_id", "title", "user_520_rating"]
    ]
)

,movie_id,title,user_520_rating
0,286,"English Patient, The (1996)",5.0
1,690,Seven Years in Tibet (1997),5.0
2,898,"Postman, The (1997)",5.0
3,283,Emma (1996),4.0
4,315,Apt Pupil (1998),4.0
5,990,Anna Karenina (1997),4.0
6,289,Evita (1996),4.0
7,300,Air Force One (1997),4.0
8,310,"Rainmaker, The (1997)",4.0


In [34]:
import numpy as np

top_similar_users = similar_users.head(10)

print("Selected similar users:")
display(top_similar_users)

Selected similar users:


,1
user_id,
520,0.918559
107,0.916534
105,0.882299
510,0.862582
803,0.849655
791,0.845489
572,0.843661
691,0.838742
754,0.810643


In [35]:
import numpy as np

neighbor_ratings = rating_matrix.loc[top_similar_users.index]

weighted_sum = neighbor_ratings.fillna(0).T.dot(top_similar_users)

weighted_denominator = neighbor_ratings.notna().T.dot(top_similar_users)

movie_scores = weighted_sum / weighted_denominator.replace(0, np.nan)

movie_scores = movie_scores.drop(
    labels=list(user_1_movies),
    errors="ignore"
)

movie_scores = movie_scores.dropna().sort_values(ascending=False)

print("Number of movies available for recommendation:", len(movie_scores))
print("\nTop 10 recommended movie IDs and their predicted ratings:")
display(movie_scores.head(10))

Number of movies available for recommendation: 118

Top 10 recommended movie IDs and their predicted ratings:


/tmp/ipykernel_4896/3197084364.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  movie_scores = weighted_sum / weighted_denominator.replace(0, np.nan)


,0
movie_id,
276,5.0
318,5.0
496,5.0
524,5.0
604,5.0
477,5.0
650,5.0
735,5.0
692,5.0


In [36]:

top_recommendations = movie_scores.head(10).rename(
    "predicted_rating"
).reset_index()

top_recommendations = top_recommendations.merge(
    movie_titles,
    on="movie_id",
    how="left"
)

display(
    top_recommendations[
        ["movie_id", "title", "predicted_rating"]
    ]
)

,movie_id,title,predicted_rating
0,276,Leaving Las Vegas (1995),5.0
1,318,Schindler's List (1993),5.0
2,496,It's a Wonderful Life (1946),5.0
3,524,"Great Dictator, The (1940)",5.0
4,604,It Happened One Night (1934),5.0
5,477,Matilda (1996),5.0
6,650,"Seventh Seal, The (Sjunde inseglet, Det) (1957)",5.0
7,735,Philadelphia (1993),5.0
8,692,"American President, The (1995)",5.0
9,603,Rear Window (1954),5.0


In [37]:
print("Ratings dataset shape:", ratings.shape)
print("\nColumn names:")
print(ratings.columns.tolist())

display(ratings.head())

Ratings dataset shape: (100000, 4)

Column names:
['user_id', 'movie_id', 'rating', 'timestamp']


,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [38]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    ratings,
    test_size=0.2,
    random_state=42
)

print("Total ratings:", len(ratings))
print("Training ratings:", len(train_data))
print("Testing ratings:", len(test_data))

print("\nTraining data sample:")
display(train_data.head())

print("\nTesting data sample:")
display(test_data.head())

Total ratings: 100000
Training ratings: 80000
Testing ratings: 20000

Training data sample:


,user_id,movie_id,rating,timestamp
75220,807,1411,1,893082619
48955,474,659,5,887925187
44966,463,268,4,877384940
13568,139,286,4,879537844
92727,621,751,4,883799651



Testing data sample:


,user_id,movie_id,rating,timestamp
75721,877,381,4,882677345
80184,815,602,3,878694269
19864,94,431,4,891721716
76699,416,875,2,876696938
92991,500,182,2,883873556


In [39]:

train_matrix = train_data.pivot_table(
    index="user_id",
    columns="movie_id",
    values="rating"
)

print("Training matrix shape:", train_matrix.shape)
print("Number of users:", train_matrix.shape[0])
print("Number of movies:", train_matrix.shape[1])

display(train_matrix.head())

Training matrix shape: (943, 1653)
Number of users: 943
Number of movies: 1653


movie_id,1,2,3,4,5,6,7,8,9,10,...,1668,1670,1671,1672,1673,1676,1678,1679,1680,1681
user_id,,,,,,,,,,,,,,,,,,,,,
1,NaN,3.0,4.0,NaN,3.0,NaN,4.0,NaN,5.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [40]:

train_similarity = train_matrix.T.corr(
    method="pearson",
    min_periods=5
)

print("Similarity matrix shape:", train_similarity.shape)

display(train_similarity.iloc[:5, :5].round(2))

Similarity matrix shape: (943, 943)


user_id,1,2,3,4,5
user_id,,,,,
1,1.00,-0.02,-0.28,NaN,0.28
2,-0.02,1.00,-0.06,0.07,0.33
3,-0.28,-0.06,1.00,-0.22,NaN
4,NaN,0.07,-0.22,1.00,NaN
5,0.28,0.33,NaN,NaN,1.00


In [41]:

target_user = 1

similar_users_eval = train_similarity.loc[target_user].drop(
    labels=[target_user]
)

similar_users_eval = similar_users_eval.dropna()
similar_users_eval = similar_users_eval[
    similar_users_eval > 0
]

top_neighbors_eval = similar_users_eval.sort_values(
    ascending=False
).head(10)

print("Target user:", target_user)
print("Number of similar users selected:", len(top_neighbors_eval))
print("\nTop 10 similar users:")
display(top_neighbors_eval)

Target user: 1
Number of similar users selected: 10

Top 10 similar users:


,1
user_id,
482,0.968246
572,0.963087
105,0.952579
571,0.922613
803,0.910713
225,0.888523
93,0.878310
701,0.876460
550,0.875000


In [42]:

neighbor_ratings_eval = train_matrix.loc[top_neighbors_eval.index]

weighted_sum_eval = neighbor_ratings_eval.fillna(0).T.dot(
    top_neighbors_eval
)

weighted_denominator_eval = neighbor_ratings_eval.notna().T.dot(
    top_neighbors_eval
)

movie_scores_eval = (
    weighted_sum_eval
    / weighted_denominator_eval.replace(0, np.nan)
)

user_seen_movies = train_matrix.loc[target_user].dropna().index

movie_scores_eval = movie_scores_eval.drop(
    labels=user_seen_movies,
    errors="ignore"
)

movie_scores_eval = movie_scores_eval.dropna().sort_values(
    ascending=False
)

top_10_eval = movie_scores_eval.head(10)

print("Top 10 recommendations for User 1:")
display(top_10_eval)

Top 10 recommendations for User 1:


/tmp/ipykernel_4896/3427669477.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  / weighted_denominator_eval.replace(0, np.nan)


,0
movie_id,
100,5.0
147,5.0
285,5.0
305,5.0
302,5.0
316,5.0
310,5.0
323,5.0
928,5.0


In [43]:

user_test_ratings = test_data[
    test_data["user_id"] == target_user
]

relevant_movies = set(
    user_test_ratings.loc[
        user_test_ratings["rating"] >= 4,
        "movie_id"
    ]
)

recommended_movies = set(top_10_eval.index)

relevant_recommendations = (
    recommended_movies & relevant_movies
)

print("User:", target_user)
print("Movies in the top 10 recommendations:", len(recommended_movies))
print("Relevant movies in the test set:", len(relevant_movies))
print("Relevant movies recommended:", len(relevant_recommendations))

print("\nRelevant recommended movie IDs:")
print(sorted(relevant_recommendations))

User: 1
Movies in the top 10 recommendations: 10
Relevant movies in the test set: 34
Relevant movies recommended: 1

Relevant recommended movie IDs:
[100]


In [44]:

precision_at_10 = (
    len(relevant_recommendations) / len(recommended_movies)
)

recall_at_10 = (
    len(relevant_recommendations) / len(relevant_movies)
    if relevant_movies else 0
)

print(f"Precision@10: {precision_at_10:.4f}")
print(f"Recall@10: {recall_at_10:.4f}")

Precision@10: 0.1000
Recall@10: 0.0294


In [45]:

if precision_at_10 + recall_at_10 > 0:
    f1_at_10 = (
        2 * precision_at_10 * recall_at_10
        / (precision_at_10 + recall_at_10)
    )
else:
    f1_at_10 = 0

print(f"F1-score@10: {f1_at_10:.4f}")

F1-score@10: 0.0455


In [46]:

ranked_recommendations = list(top_10_eval.index)

precision_at_relevant_positions = []

relevant_count = 0

for rank, movie_id in enumerate(ranked_recommendations, start=1):
    if movie_id in relevant_movies:
        relevant_count += 1
        precision_at_rank = relevant_count / rank
        precision_at_relevant_positions.append(precision_at_rank)

if relevant_movies:
    average_precision_at_10 = (
        sum(precision_at_relevant_positions)
        / min(len(relevant_movies), 10)
    )
else:
    average_precision_at_10 = 0

print("Relevant movie positions:", [
    rank for rank, movie_id in enumerate(ranked_recommendations, start=1)
    if movie_id in relevant_movies
])

print(f"Average Precision@10: {average_precision_at_10:.4f}")

Relevant movie positions: [1]
Average Precision@10: 0.1000


In [47]:
import numpy as np

relevance_scores = [
    1 if movie_id in relevant_movies else 0
    for movie_id in ranked_recommendations
]

discounts = np.log2(np.arange(2, len(relevance_scores) + 2))
dcg = np.sum(np.array(relevance_scores) / discounts)

ideal_relevance = sorted(relevance_scores, reverse=True)
idcg = np.sum(np.array(ideal_relevance) / discounts)

ndcg_at_10 = dcg / idcg if idcg > 0 else 0

print(f"NDCG@10: {ndcg_at_10:.4f}")

NDCG@10: 1.0000


In [48]:
import numpy as np
import pandas as pd

K = 10
MIN_SIMILARITY = 0

evaluation_results = []
evaluation_users = test_data["user_id"].unique()

for user_id in evaluation_users:

    if user_id not in train_matrix.index:
        continue

    similarities = train_similarity.loc[user_id].drop(
        labels=[user_id], errors="ignore"
    ).dropna()

    similarities = similarities[similarities > MIN_SIMILARITY]
    neighbors = similarities.sort_values(ascending=False).head(10)

    if neighbors.empty:
        continue

    neighbor_ratings = train_matrix.loc[neighbors.index]

    weighted_sum = neighbor_ratings.fillna(0).T.dot(neighbors)
    weighted_denominator = neighbor_ratings.notna().T.dot(neighbors)

    weighted_denominator = weighted_denominator.astype(float).mask(
        weighted_denominator == 0
    )

    predicted_scores = weighted_sum / weighted_denominator
    predicted_scores = predicted_scores.dropna()

    seen_movies = train_matrix.loc[user_id].dropna().index

    predicted_scores = predicted_scores.drop(
        labels=seen_movies, errors="ignore"
    )

    recommendations = predicted_scores.sort_values(
        ascending=False
    ).head(K).index.tolist()

    user_test = test_data[test_data["user_id"] == user_id]

    relevant_movies = set(
        user_test.loc[user_test["rating"] >= 4, "movie_id"]
    )

    if not relevant_movies:
        continue

    relevant_flags = [
        1 if movie in relevant_movies else 0
        for movie in recommendations
    ]

    hits = sum(relevant_flags)

    # Precision@10
    precision = hits / K

    # Recall@10
    recall = hits / len(relevant_movies)

    # F1-score@10
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0 else 0
    )

    # Average Precision@10
    precision_sum = 0
    relevant_count = 0

    for rank, is_relevant in enumerate(relevant_flags, start=1):
        if is_relevant:
            relevant_count += 1
            precision_sum += relevant_count / rank

    average_precision = precision_sum / min(len(relevant_movies), K)

    # NDCG@10 - corrected calculation
    discounts = 1 / np.log2(np.arange(2, K + 2))

    dcg = np.sum(
        np.array(relevant_flags) * discounts[:len(relevant_flags)]
    )

    ideal_relevant_count = min(len(relevant_movies), K)
    idcg = np.sum(discounts[:ideal_relevant_count])

    ndcg = dcg / idcg if idcg > 0 else 0

    evaluation_results.append({
        "user_id": user_id,
        "Precision@10": precision,
        "Recall@10": recall,
        "F1-score@10": f1,
        "MAP@10": average_precision,
        "NDCG@10": ndcg
    })

evaluation_df = pd.DataFrame(evaluation_results)

if not evaluation_df.empty:

    print("Number of users evaluated:", len(evaluation_df))
    print("\nAverage evaluation metrics:")

    display(
        evaluation_df[
            [
                "Precision@10",
                "Recall@10",
                "F1-score@10",
                "MAP@10",
                "NDCG@10"
            ]
        ].mean().to_frame("Score").round(4)
    )

else:
    print("No users could be evaluated. Check the training and testing data.")

Number of users evaluated: 920

Average evaluation metrics:


,Score
Precision@10,0.0342
Recall@10,0.0226
F1-score@10,0.0232
MAP@10,0.0157
NDCG@10,0.0393
